# Online Retail Customer Segmentation

A consolidated version of two coursework notebooks. It cleans transactions, creates customer-level RFM features, evaluates K-Means choices, inspects DBSCAN outliers, and profiles segments. Dataset source: [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail), CC BY 4.0. Cluster names are descriptive, not verified customer types.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

DATA_FILE = Path("data/Online Retail.xlsx")
if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Place the UCI workbook at {DATA_FILE}. See README.")
raw = pd.read_excel(DATA_FILE)
required = {"InvoiceNo", "InvoiceDate", "CustomerID", "Quantity", "UnitPrice", "Country"}
if not required.issubset(raw.columns):
    raise ValueError(f"Missing columns: {sorted(required - set(raw.columns))}")
print("Original transactions:", len(raw))


## 1. Clean transactions and create RFM

Remove cancellations, missing customer IDs, and nonpositive quantities or prices. Monetary means total observed line revenue; frequency counts unique invoices. The reference date is one day after the latest valid transaction.


In [ ]:
tx = raw.loc[~raw["InvoiceNo"].astype(str).str.upper().str.startswith("C")].copy()
tx = tx.dropna(subset=["CustomerID", "InvoiceDate"])
tx = tx[(tx["Quantity"] > 0) & (tx["UnitPrice"] > 0)].copy()
tx["InvoiceDate"] = pd.to_datetime(tx["InvoiceDate"])
tx["Revenue"] = tx["Quantity"] * tx["UnitPrice"]
reference_date = tx["InvoiceDate"].max() + pd.Timedelta(days=1)
customers = tx.groupby("CustomerID").agg(
    LastPurchase=("InvoiceDate", "max"),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum"),
    Country=("Country", lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0]),
)
customers["Recency"] = (reference_date - customers["LastPurchase"]).dt.days
customers = customers.reset_index()
print(f"Valid transaction lines: {len(tx)} | Customers: {len(customers)}")
print(customers[["Recency", "Frequency", "Monetary"]].describe().round(2))


## 2. Prepare features and inspect K-Means choices

Apply `log1p` to skewed frequency and monetary values, then scale RFM features. Silhouette is diagnostic: a high score alone can select a solution with tiny, impractical groups. We require every cluster to contain at least 1% of customers when choosing a candidate.


In [ ]:
features = customers[["Recency", "Frequency", "Monetary"]].copy()
features["Frequency"] = np.log1p(features["Frequency"])
features["Monetary"] = np.log1p(features["Monetary"])
X = StandardScaler().fit_transform(features)
min_cluster_size = max(2, int(np.ceil(0.01 * len(customers))))
rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    rows.append({"k":k, "inertia":km.inertia_,
                 "silhouette":silhouette_score(X, labels, sample_size=min(1500, len(X)), random_state=42),
                 "smallest_cluster":np.bincount(labels).min()})
selection = pd.DataFrame(rows)
print(selection.round(3))
eligible = selection[selection["smallest_cluster"] >= min_cluster_size]
if eligible.empty:
    raise ValueError("No K=2..8 met the 1% minimum-cluster rule; inspect the data and criteria.")
best_k = int(eligible.sort_values(["silhouette", "k"], ascending=[False, True]).iloc[0]["k"])
print("Selected k:", best_k)


In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
customers["Cluster"] = kmeans.fit_predict(X)
profile = customers.groupby("Cluster").agg(
    Customers=("CustomerID", "size"),
    MeanRecency=("Recency", "mean"),
    MeanFrequency=("Frequency", "mean"),
    MeanMonetary=("Monetary", "mean"),
    TotalRevenue=("Monetary", "sum"),
).round(2)
profile["RevenueSharePct"] = (100 * profile["TotalRevenue"] / profile["TotalRevenue"].sum()).round(2)
print(profile)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=profile.index.astype(str), y=profile["Customers"], ax=axes[0], color="#738ac2")
sns.barplot(x=profile.index.astype(str), y=profile["RevenueSharePct"], ax=axes[1], color="#d68199")
axes[0].set(xlabel="Cluster", ylabel="Customers")
axes[1].set(xlabel="Cluster", ylabel="Revenue share (%)")
plt.tight_layout()
plt.show()


## 3. DBSCAN sensitivity check

The 90th percentile of fifth-neighbor distance is only a starting point for `eps`. Report the cluster/noise counts instead of labeling the largest DBSCAN group as “top customers.” DBSCAN labels are not directly comparable to K-Means labels.


In [ ]:
min_samples = 5
neighbors = NearestNeighbors(n_neighbors=min_samples).fit(X)
distances, _ = neighbors.kneighbors(X)
eps_guess = float(np.percentile(distances[:, -1], 90))
for multiplier in (0.8, 1.0, 1.2):
    labels = DBSCAN(eps=eps_guess * multiplier, min_samples=min_samples).fit_predict(X)
    counts = pd.Series(labels).value_counts().sort_index()
    print(f"eps={eps_guess * multiplier:.3f}; groups={sum(counts.index >= 0)}; noise={counts.get(-1, 0)}")
    print(counts.to_dict())


## Interpretation

Inspect RFM profiles and group sizes before attaching business names or targeting recommendations. The two original notebooks used different feature sets and choices of K, and one run yielded a cluster with only two customers. Their cluster IDs are arbitrary and their outputs should not be merged as if they were identical. This consolidated version reports its own results when run; it does not inherit a historical silhouette or revenue claim.
